## Workshop 2.2: Model Inversion and Differential Privacy

### Learning Objectives:

By the end of this notebook, you will:
- Understand how AI models are vulnerable to data leakage
- Perform a model inversion attack
- Learn how to prevent attacks through training with differential privacy

### Step 1: Load the dataset

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# We want to load the dataset as a pandas DataFrame to keep the feature names for later
iris = sns.load_dataset("iris")
print(iris.head())

# Create a pairplot to visualize the relationships between features
sns.pairplot(iris, 
            hue='species', height=2)
plt.show()

### Step 2: Train a simple model

Next we will train a simple classifier on the Iris dataset. To do this we must:

1. Split the data into train-test sets
2. Train the model using the train set
3. Evaluate the model using the test set

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Prepare the Data
# Split the data into training and testing sets (75% train, 25% test)
train, test = train_test_split(iris, test_size=0.25, random_state=0)

# Train a Model
# We will use a Multi-layer Perceptron classifier for this example
model = MLPClassifier(hidden_layer_sizes=(100), activation='relu', random_state=42)
model.fit(train.drop(columns=['species']), train['species'])

# Make Predictions
y_pred = model.predict(test.drop(columns=['species']))

# Evaluate Performance
accuracy = accuracy_score(test['species'], y_pred)
print(f"Model Accuracy: {accuracy:.2f}")


### Step 3: Model Inversion

Model inversion attacks can extract training data that was memorized by a machine learning model. 

In the first cell we will take a look at how the petal length feature influences the model.

In the second cell we will perform a model inversion attack.

A model inversion attack proceeds as follows:
1. Pick a class to extract. We will extract all three!
2. Choose a random start point as input.
3. Query the model and check the probabilities.
4. Minimize the objective function, which will maximize our confidence.

In [ ]:
import pandas as pd
import numpy as np

def feature_influence(predictor, feature_ranges, n_samples=1000):
    inferred_values = []
    # For each sample, generate random feature values within the specified ranges
    for _ in range(n_samples):
        synthetic_data = {}
        for feature, frange in feature_ranges.items():
            synthetic_data[feature] = np.random.uniform(frange[0], frange[1])
        synthetic_df = pd.DataFrame([synthetic_data])
        
        # Get the model predictions for this synthetic datapoint
        choice = predictor(synthetic_df)[0]
        inferred_values.append({
            'sepal_length': synthetic_data['sepal_length'],
            'sepal_width': synthetic_data['sepal_width'],
            'petal_length': synthetic_data['petal_length'],
            'petal_width': synthetic_data['petal_width'],
            'predicted_class': choice
        })
    inferred_values = pd.DataFrame(inferred_values)
    return inferred_values

# Attack the model to infer the 'petal length' feature
feature_ranges = {
    'sepal_length': (4.0, 7.9), 
    'sepal_width': (2.0, 4.5), 
    'petal_length': (1.0, 7.0), 
    'petal_width': (0.1, 2.5)
    }

results = feature_influence(model.predict, feature_ranges)

# Output the results
for i in range(5):
    row = results.iloc[i]
    print(f"Sampled petal length: {row['petal_length']:.2f}, Predicted class: {row['predicted_class']}")

# Plot the petal length
sns.histplot(results, x='petal_length', hue='predicted_class', bins=50, kde=True)
plt.show()

In [ ]:
from scipy.optimize import minimize

def model_inversion_attack(proba_func, target_class_index, feature_ranges, n_restarts=10):
    bounds = list(feature_ranges.values())
    best_input, best_score = None, -np.inf

    # Objective function to maximise the probability of the target class
    def objective(x):
        df = pd.DataFrame([x], columns=list(feature_ranges.keys()))
        proba = proba_func(df)[0]
        return -proba[target_class_index]  # Minimise negative = maximise confidence

    for _ in range(n_restarts):
        # Random start within feature bounds
        x0 = [np.random.uniform(lo, hi) for lo, hi in bounds]
        result = minimize(objective, x0, bounds=bounds, method='L-BFGS-B')
        if -result.fun > best_score:
            best_score = -result.fun
            best_input = result.x

    return pd.Series(best_input, index=list(feature_ranges.keys())), best_score


for i, species in enumerate(model.classes_):
    recovered, confidence = model_inversion_attack(model.predict_proba, i, feature_ranges)
    print(f"\nRecovered '{species}' (confidence: {confidence:.2f}):")
    print(recovered.round(1))    

### Step 4: Differential Privacy

Differential privacy prevents model inversion attacks by applying small targeted random noise to either the training data or the output of a model. We will implement differential privacy on the output of the model by using the exponential mechanism.

In [ ]:
import numpy as np

# Change this value to adjust privacy level, higher means less privacy but more accuracy
EPS = 3.0

# Exponential mechanism for differential privacy
def exponential(model, sample, epsilon):
    scores = model.predict_proba(sample)[0]  # Get the predicted probabilities

    probabilities = np.array([np.exp(epsilon * score / 2) for score in scores])
    probabilities = probabilities / np.sum(probabilities)  # Normalize to get a probability distribution
    
    choice = np.random.choice(len(probabilities), p=probabilities)  # Sample from the distribution
    return model.classes_[choice]  # Return the predicted class based on the sampled index

def exponential_probas(model, sample, epsilon):
    scores = model.predict_proba(sample)[0]  # Get the predicted probabilities

    probabilities = np.array([np.exp(epsilon * score / 2) for score in scores])
    probabilities = probabilities / np.sum(probabilities)  # Normalize to get a probability distribution
    
    return probabilities

# Functions to call our model with differential privacy
def private_predict(X):
    predictions = []
    for i in range(len(X)):
        sample = X.iloc[[i]]
        predictions.append(exponential(model, sample, epsilon=EPS))
    return predictions

def private_probas(X):
    all_probas = []
    for i in range(len(X)):
        sample = X.iloc[[i]]
        all_probas.append(exponential_probas(model, sample, epsilon=EPS))
    return np.array(all_probas)

# Example usage of private_predict
test_example = pd.DataFrame([test.drop(columns=['species']).iloc[0]])

example = private_predict(test_example)
print(f"Actual label for the first test sample: {test['species'].iloc[0]}")

probas = model.predict_proba(test_example)[0]
print(f"Model's predicted probabilities for the first test sample: {probas}")

mechanism_probas = probabilities = np.array([np.exp(EPS * score / 2) for score in probas])
mechanism_probas = probabilities / np.sum(probabilities)
print(f"Exponential mechanism probabilities for the first test sample: {mechanism_probas}")
print(f"Private prediction for the first test sample: {example[0]}")

# Calculate accuracy of private predictions on the test set
X_test = test.drop(columns=['species'])
private_preds = private_predict(X_test)
private_accuracy = accuracy_score(test['species'], private_preds)
print(f"Private Model Accuracy: {private_accuracy:.2f}")

### Step 5: Run the attack again

In [ ]:
# Running the attack
feature_influence_results = feature_influence(private_predict, feature_ranges)

for i, species in enumerate(model.classes_):
    recovered, confidence = model_inversion_attack(private_probas, i, feature_ranges)
    print(f"\nRecovered '{species}' (confidence: {confidence:.2f}):")
    print(recovered.round(1))

# Plot the petal length
sns.histplot(feature_influence_results, x='petal_length', hue='predicted_class', bins=50, kde=True)
plt.show()